In [1]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv("../data/DateFruit Dataset.csv")
df

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.9080,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.1440,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.3940,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.2100,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.3540,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
893,255403,1925.3650,691.8453,477.1796,0.7241,570.2536,0.9785,261028,0.7269,1.4499,...,2.2423,2.3704,2.7202,-25296416768,-19168882688,-18473392128,49.0869,43.0422,42.4153,SOGAY
894,365924,2664.8230,855.4633,551.5447,0.7644,682.5752,0.9466,386566,0.6695,1.5510,...,3.4109,3.5805,3.9910,-31605219328,-21945366528,-19277905920,46.8086,39.1046,36.5502,SOGAY
895,254330,1926.7360,747.4943,435.6219,0.8126,569.0545,0.9925,256255,0.7240,1.7159,...,2.2759,2.5090,2.6951,-22242772992,-19594921984,-17592152064,44.1325,40.7986,40.9769,SOGAY
896,238955,1906.2679,716.6485,441.8297,0.7873,551.5859,0.9604,248795,0.6954,1.6220,...,2.6769,2.6874,2.7991,-26048595968,-21299822592,-19809978368,51.2267,45.7162,45.6260,SOGAY


In [7]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

X = df.drop("Class", axis=1)
y = df["Class"]

lab_encoder = LabelEncoder()
y = lab_encoder.fit_transform(y)

In [8]:
# Splitting the data

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42, test_size=0.2
)

In [9]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ANN Classifier

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [11]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)    # can skip .values as it is already a number

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)   # The loss function expects target values in long format


In [12]:
# Create tensor dataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Load the dataset
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [18]:
# Building the model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 7)
        )

    def forward(self, x):
        return self.model(x)

In [19]:
model = ANN()

# loss and optimiser

criterion = nn.CrossEntropyLoss()
optimiser = optim.Adam(model.parameters())


In [20]:
# Training

epochs = 100

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for xb, yb in train_loader:
        optimiser.zero_grad()

        predictions = model(xb)
        loss = criterion(predictions, yb)
        loss.backward()
        optimiser.step()
        running_loss += loss

    train_loss = running_loss / len(train_loader)

    print(f"Epoch: {epoch+1}/{epochs} => loss: {train_loss}")


Epoch: 1/100 => loss: 1.6880589723587036
Epoch: 2/100 => loss: 1.0850512981414795
Epoch: 3/100 => loss: 0.7031558752059937
Epoch: 4/100 => loss: 0.5359174609184265
Epoch: 5/100 => loss: 0.44306397438049316
Epoch: 6/100 => loss: 0.3933204710483551
Epoch: 7/100 => loss: 0.34502527117729187
Epoch: 8/100 => loss: 0.3081354796886444
Epoch: 9/100 => loss: 0.2790793478488922
Epoch: 10/100 => loss: 0.2582811117172241
Epoch: 11/100 => loss: 0.23407307267189026
Epoch: 12/100 => loss: 0.22213038802146912
Epoch: 13/100 => loss: 0.22107769548892975
Epoch: 14/100 => loss: 0.20464111864566803
Epoch: 15/100 => loss: 0.19406673312187195
Epoch: 16/100 => loss: 0.18555636703968048
Epoch: 17/100 => loss: 0.17468535900115967
Epoch: 18/100 => loss: 0.19080325961112976
Epoch: 19/100 => loss: 0.1699845790863037
Epoch: 20/100 => loss: 0.16133122146129608
Epoch: 21/100 => loss: 0.15164418518543243
Epoch: 22/100 => loss: 0.14941520988941193
Epoch: 23/100 => loss: 0.14017432928085327
Epoch: 24/100 => loss: 0.1395

In [31]:
# Evaluation

model.eval()
correct = 0 
total = 0

with torch.no_grad():
    for xb, yb in test_loader:
        predictions = model(xb)
        _, predicted = torch.max(predictions, 1)    # torch.max returns the max val and it index(its actual category)

        total += yb.size(0)
        correct += (predicted == yb).sum().item()

print(f"total: {total}, correct: {correct}")
print(f"Accuracy: {correct/total:.4f}")


total: 180, correct: 168
Accuracy: 0.9333
